In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import pickle


In [10]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
df = df.iloc[:,3:]

In [12]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [13]:
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.sample(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
6467,540,France,0,31,7,0.00,1,0,1,183051.60,1
5831,635,France,0,27,8,127471.56,1,1,1,152916.05,1
7162,607,Spain,1,34,9,132439.99,1,1,0,177747.72,0
1137,583,France,0,42,4,0.00,2,1,0,17439.66,0
9262,734,Germany,0,52,6,71283.09,2,0,1,38984.37,0


In [25]:
df['Geography'].unique()

<ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [17]:
label_encoder_gender.classes_

array(['Female', 'Male'], dtype=object)

In [35]:
ohe_geo = OneHotEncoder()
geography = ohe_geo.fit_transform(df[['Geography']])


In [36]:
ohe_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [41]:
geography.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [44]:
geo_df = pd.DataFrame(data = geography.toarray(), columns=ohe_geo.get_feature_names_out())

In [49]:
df = pd.concat([df , geo_df] , axis=1)

In [50]:
df = df.drop(columns = ['Geography'])

In [51]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [52]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender , file)

with open("onehotencoder_geo.pkl" , 'wb') as file:
    pickle.dump(ohe_geo , file)

In [53]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [56]:
X = df.drop(columns = ['Exited'])
y = df['Exited']

In [57]:
x_train , x_test , y_train , y_test  = train_test_split(X, y , test_size=0.2 , random_state=3)

In [58]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [59]:
with open('scaler.pkl' , 'wb') as file:
    pickle.dump(scaler , file)

In [61]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard
import datetime

In [62]:
x_train.shape

(8000, 12)

In [88]:
model  = Sequential([
    Dense(units=64 , activation='relu' , input_shape = (12,)),
    Dense(units=32 , activation='relu'),
    Dense(units=1 , activation='sigmoid')
])

c:\Users\M S I\Desktop\ANN_Classification_Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [89]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()
metric = tf.keras.metrics.Accuracy()

In [91]:
model.compile(optimizer=opt , loss=loss , metrics=[metric])

In [92]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir =log_dir , histogram_freq=1)

In [93]:
early_stopping_callback = EarlyStopping(monitor='val_loss' , patience=10 , restore_best_weights=True)


In [94]:
history = model.fit(
    x_train , y_train , validation_data = (x_test , y_test) , epochs = 100,
    callbacks = [tensorflow_callback , early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0000e+00 - loss: 0.3866 - val_accuracy: 0.0000e+00 - val_loss: 0.3587
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 0.3554 - val_accuracy: 0.0000e+00 - val_loss: 0.3638
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 0.3496 - val_accuracy: 0.0000e+00 - val_loss: 0.3625
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 0.3442 - val_accuracy: 0.0000e+00 - val_loss: 0.3481
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.2500e-04 - loss: 0.3395 - val_accuracy: 0.0000e+00 - val_loss: 0.3526
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: 0.3381 - val_accuracy: 0.0000e+00 - val_loss: 0.3573
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 0.3322 - val_accuracy: 0.0000e+00 - val_loss: 0.3547
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 

In [95]:
model.save("model.h5")

In [96]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [99]:
%tensorboard --logdir logs/fit/20260512-151242

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\M S I\Desktop\ANN_Classification_Project\.venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
    from tensorboard.main import run_main
  File "C:\Users\M S I\Desktop\ANN_Classification_Project\.venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\Users\M S I\Desktop\ANN_Classification_Project\.venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [107]:
df.head(1).to_dict()

{'CreditScore': {0: 619},
 'Gender': {0: 0},
 'Age': {0: 42},
 'Tenure': {0: 2},
 'Balance': {0: 0.0},
 'NumOfProducts': {0: 1},
 'HasCrCard': {0: 1},
 'IsActiveMember': {0: 1},
 'EstimatedSalary': {0: 101348.88},
 'Exited': {0: 1},
 'Geography_France': {0: 1.0},
 'Geography_Germany': {0: 0.0},
 'Geography_Spain': {0: 0.0}}